<a href="https://colab.research.google.com/github/igorfantucci/Aula-Automatica---GRUPO-5/blob/main/etapa-03-arvores/Aula%2027%20-%20%C3%81rvores%20de%20Decis%C3%A3o%20-%20Algoritmos%20Cl%C3%A1ssicos%20de%20Constru%C3%A7%C3%A3o%20de%20%C3%81rvore%20-%20code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 27: Árvores de Decisão — Algoritmos Clássicos (ID3, C4.5 e CART) na Planta de Biodiesel (Grupo 5)

Este caderno implementa **do zero em Python** um construtor clássico de Árvores de Decisão com suporte aos critérios **Índice de Gini (CART)** e **Entropia / Ganho de Informação (ID3/C4.5)**, controle de pré-poda (`max_depth`, `min_samples_split`), extração automática de regras `IF...THEN...ELSE` para CLP (IEC 61131-3) e avaliação em dados simulados da **Planta Piloto de Produção de Biodiesel**.

In [ ]:
import math
import random
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

## 1. Implementação do zero: Critérios de Impureza (Gini & Entropia) e Construtor CART/C4.5

In [ ]:
class DecisionNode:
    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, *, value=None, impurity=0.0, n_samples=0, counts=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.impurity = impurity
        self.n_samples = n_samples
        self.counts = counts or {}

    def is_leaf(self):
        return self.value is not None


class BiodieselDecisionTree:
    def __init__(self, criterion='gini', max_depth=4, min_samples_split=2):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
        self.feature_names = []
        self.feature_importances_ = []

    def _impurity(self, y):
        n = len(y)
        if n == 0:
            return 0.0
        counts = Counter(y)
        probs = [c / n for c in counts.values()]
        if self.criterion == 'gini':
            return 1.0 - sum(p ** 2 for p in probs)
        elif self.criterion == 'entropy':
            return -sum(p * math.log2(p) for p in probs if p > 0)
        raise ValueError(f'Critério desconhecido: {self.criterion}')

    def _best_split(self, X, y):
        n_samples = len(y)
        n_features = len(X[0])
        parent_imp = self._impurity(y)
        best_gain = -1.0
        best_feat, best_thresh = None, None

        for feat_idx in range(n_features):
            paired = sorted(zip((row[feat_idx] for row in X), y), key=lambda t: t[0])
            for i in range(1, n_samples):
                if paired[i - 1][0] == paired[i][0]:
                    continue
                if paired[i - 1][1] == paired[i][1]:
                    # Otimização: limiar ótimo ocorre em fronteiras de mudança de classe
                    continue
                thresh = 0.5 * (paired[i - 1][0] + paired[i][0])
                y_left = [lbl for val, lbl in paired[:i]]
                y_right = [lbl for val, lbl in paired[i:]]
                w_left = len(y_left) / n_samples
                w_right = len(y_right) / n_samples
                gain = parent_imp - (w_left * self._impurity(y_left) + w_right * self._impurity(y_right))
                if gain > best_gain:
                    best_gain = gain
                    best_feat = feat_idx
                    best_thresh = thresh

        return best_feat, best_thresh, best_gain

    def _build(self, X, y, depth):
        counts = dict(Counter(y))
        imp = self._impurity(y)
        n_samples = len(y)
        majority_class = Counter(y).most_common(1)[0][0]

        if depth >= self.max_depth or n_samples < self.min_samples_split or len(counts) == 1:
            return DecisionNode(value=majority_class, impurity=imp, n_samples=n_samples, counts=counts)

        feat_idx, thresh, gain = self._best_split(X, y)
        if feat_idx is None or gain <= 1e-9:
            return DecisionNode(value=majority_class, impurity=imp, n_samples=n_samples, counts=counts)

        self.feature_importances_[feat_idx] += gain * n_samples
        left_idx = [i for i, row in enumerate(X) if row[feat_idx] <= thresh]
        right_idx = [i for i, row in enumerate(X) if row[feat_idx] > thresh]

        left_child = self._build([X[i] for i in left_idx], [y[i] for i in left_idx], depth + 1)
        right_child = self._build([X[i] for i in right_idx], [y[i] for i in right_idx], depth + 1)
        return DecisionNode(feat_idx, thresh, left_child, right_child, impurity=imp, n_samples=n_samples, counts=counts)

    def fit(self, X, y, feature_names=None):
        n_features = len(X[0])
        self.feature_names = feature_names or [f'X{i}' for i in range(n_features)]
        self.feature_importances_ = [0.0] * n_features
        self.root = self._build(X, y, depth=0)
        total_imp = sum(self.feature_importances_)
        if total_imp > 0:
            self.feature_importances_ = [v / total_imp for v in self.feature_importances_]
        return self

    def _predict_one(self, row, node):
        if node.is_leaf():
            return node.value
        if row[node.feature_idx] <= node.threshold:
            return self._predict_one(row, node.left)
        return self._predict_one(row, node.right)

    def predict(self, X):
        return [self._predict_one(row, self.root) for row in X]

    def export_plc_rules(self, node=None, indent=''):
        if node is None:
            node = self.root
        if node.is_leaf():
            return f"{indent}DIAGNOSTICO := '{node.value}'; // n={node.n_samples}, imp={node.impurity:.3f}\n"
        fname = self.feature_names[node.feature_idx]
        code = f"{indent}IF {fname} <= {node.threshold:.2f} THEN\n"
        code += self.export_plc_rules(node.left, indent + '    ')
        code += f"{indent}ELSE // {fname} > {node.threshold:.2f}\n"
        code += self.export_plc_rules(node.right, indent + '    ')
        code += f"{indent}END_IF;\n"
        return code

## 2. Geração do Dataset da Planta de Biodiesel e Comparativo Gini (CART) vs. Entropia (C4.5)

In [ ]:
feature_names = ['TT-201 (°C)', 'PT-201 (bar)', 'LT-201 (%)', 'AT-100 (ppm)', 'IT-301 (u.a.)']

# Geração de 400 bateladas simuladas com ruído realista de instrumentação
specs = [
    ('NORMAL_B100',         160, [57.5, 1.05, 65.0,  2.0, 0.85], [1.5, 0.06, 10.0, 1.2, 0.06]),
    ('RISCO_FLASH_BOILING',  90, [66.5, 2.65, 65.0,  9.0, 0.80], [1.8, 0.28,  8.0, 3.0, 0.08]),
    ('VAZAMENTO_METANOL',    75, [57.0, 1.05, 60.0, 26.0, 0.82], [1.5, 0.05,  9.0, 4.5, 0.07]),
    ('FALHA_MEC_SEPARACAO',  75, [57.0, 1.03, 12.5,  2.5, 0.45], [2.0, 0.05,  2.5, 1.2, 0.12]),
]

X_all, y_all = [], []
for cls, n, means, stds in specs:
    pts = np.random.normal(means, stds, size=(n, 5))
    pts[:, 1] = np.clip(pts[:, 1], 0.8, 4.0)
    pts[:, 2] = np.clip(pts[:, 2], 2.0, 98.0)
    pts[:, 3] = np.clip(pts[:, 3], 0.0, 60.0)
    pts[:, 4] = np.clip(pts[:, 4], 0.05, 1.0)
    for row in pts:
        X_all.append([round(float(v), 3) for v in row])
        y_all.append(cls)

# Embaralhamento e Split Treino (70%) / Teste (30%)
indices = list(range(len(X_all)))
random.shuffle(indices)
split_idx = int(0.70 * len(indices))
train_idx, test_idx = indices[:split_idx], indices[split_idx:]

X_train = [X_all[i] for i in train_idx]
y_train = [y_all[i] for i in train_idx]
X_test  = [X_all[i] for i in test_idx]
y_test  = [y_all[i] for i in test_idx]

for crit in ['gini', 'entropy']:
    clf = BiodieselDecisionTree(criterion=crit, max_depth=4, min_samples_split=4)
    clf.fit(X_train, y_train, feature_names=feature_names)
    acc_tr = sum(p == t for p, t in zip(clf.predict(X_train), y_train)) / len(y_train)
    acc_te = sum(p == t for p, t in zip(clf.predict(X_test), y_test)) / len(y_test)
    print(f'Critério={crit.upper():7s} | Acurácia Treino={acc_tr*100:.2f}% | Acurácia Teste={acc_te*100:.2f}%')

print('\n--- Código IEC 61131-3 (Structured Text) Gerado pelo CART (Gini) ---')
clf_gini = BiodieselDecisionTree(criterion='gini', max_depth=4, min_samples_split=4).fit(X_train, y_train, feature_names)
print(clf_gini.export_plc_rules())

## 3. Análise de *Overfitting* (`max_depth`) e Importância das Variáveis de Processo

In [ ]:
depths = list(range(1, 9))
acc_train_d, acc_test_d = [], []
for d in depths:
    m = BiodieselDecisionTree(criterion='gini', max_depth=d, min_samples_split=2).fit(X_train, y_train, feature_names)
    acc_train_d.append(100 * sum(p == t for p, t in zip(m.predict(X_train), y_train)) / len(y_train))
    acc_test_d.append(100 * sum(p == t for p, t in zip(m.predict(X_test), y_test)) / len(y_test))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))

ax1.plot(depths, acc_train_d, 'o-', color='#10b981', label='Acurácia Treino')
ax1.plot(depths, acc_test_d, 's--', color='#4f46e5', label='Acurácia Teste')
ax1.set_xlabel('Profundidade Máxima (max_depth)')
ax1.set_ylabel('Acurácia (%)')
ax1.set_title('Curva de Complexidade vs. Generalização (CART)')
ax1.legend()
ax1.grid(alpha=0.3)

y_pos = np.arange(len(feature_names))
ax2.barh(y_pos, [v * 100 for v in clf_gini.feature_importances_], color='#f59e0b', edgecolor='#b45309')
ax2.set_yticks(y_pos)
ax2.set_yticklabels(feature_names)
ax2.invert_yaxis()
ax2.set_xlabel('Importância Relativa na Redução de Impureza (%)')
ax2.set_title('Importância das Tags ISA-5.1 no Diagnóstico')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()